In [1]:
import sys 
sys.path.append("..")
import pandas as pd
from src.rag import RAGPipeline
from src.dataset import load_golden

rag=RAGPipeline()
print(rag.build_index(), "chunks indexed") 
rows=load_golden()
print(len(rows))



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

20 chunks indexed
24


In [2]:
records = []
for r in rows:
    retrieved = rag.retrieve(r.question)
    sources = [x["source"] for x in retrieved]
    top_sim = retrieved[0]["similarity"]

    if r.answerable:
        hit = any(s in sources for s in r.expected_sources)
        rank = next((i + 1 for i, s in enumerate(sources) if s in r.expected_sources), None)
    else:
        hit, rank = None, None

    records.append({
        "id": r.id,
        "type": r.question_type,
        "question": r.question,
        "expected": r.expected_source,
        "retrieved": sources,
        "hit": hit,
        "rank": rank,
        "top_similarity": top_sim,
    })

df = pd.DataFrame(records)
answerable = df[df["hit"].notna()]
print("Hit rate @3:", round(answerable["hit"].astype(float).mean(), 3))
print("Hit rate by type:\n", answerable.groupby("type")["hit"].apply(lambda s: round(s.astype(float).mean(), 2)))
df

Hit rate @3: 0.95
Hit rate by type:
 type
multi_fact     1.00
paraphrased    1.00
single_fact    0.92
Name: hit, dtype: float64


,id,type,question,expected,retrieved,hit,rank,top_similarity
0,Q001,single_fact,What is the refund period?,refund_policy.txt,"[refund_policy.txt, refund_policy.txt, refund_...",True,1.0,0.611
1,Q002,single_fact,Are digital products refundable?,refund_policy.txt,"[refund_policy.txt, refund_policy.txt, warrant...",True,1.0,0.592
2,Q003,multi_fact,How long does express shipping take and what d...,shipping_policy.txt,"[shipping_policy.txt, shipping_policy.txt, sup...",True,1.0,0.647
3,Q004,single_fact,Which countries do you ship to?,shipping_policy.txt,"[shipping_policy.txt, shipping_policy.txt, sup...",True,1.0,0.351
4,Q005,paraphrased,Can I get my money back if my order arrives br...,refund_policy.txt,"[refund_policy.txt, refund_policy.txt, warrant...",True,1.0,0.500
5,Q006,paraphrased,If I spend $80 how much is shipping?,shipping_policy.txt,"[shipping_policy.txt, shipping_policy.txt, ref...",True,1.0,0.394
6,Q007,out_of_scope,Do you sell gift cards?,,"[privacy_policy.txt, shipping_policy.txt, refu...",None,NaN,0.210
7,Q008,single_fact,What is the cost of standard shipping?,shipping_policy.txt,"[shipping_policy.txt, shipping_policy.txt, ref...",True,1.0,0.530
8,Q009,single_fact,What is the cost of Express shipping?,shipping_policy.txt,"[shipping_policy.txt, shipping_policy.txt, ref...",True,1.0,0.572
9,Q010,single_fact,Do you ship to Bangalore?,shipping_policy.txt,"[shipping_policy.txt, shipping_policy.txt, sup...",True,1.0,0.572


In [3]:
display(df[df["hit"] == False][["id", "question", "expected", "retrieved", "top_similarity"]])

,id,question,expected,retrieved,top_similarity
20,Q021,Where can i find the order details?,account_security.txt,"[shipping_policy.txt, shipping_policy.txt, sup...",0.299


In [ ]:
print("Answerable,top Similarity")
print(df[df["hit"].notna()]["top_similarity"].describe().round(3))

print("\n Out of scope ,top similarity")
display(df[df["type"] == "out_of_scope"][["question","top_similarity"]])

Answerable,top Similarity
count    20.000
mean      0.509
std       0.096
min       0.299
25%       0.446
50%       0.538
75%       0.577
max       0.647
Name: top_similarity, dtype: float64

 Out of scope ,top similarity


KeyError: ('question', 'top_similarity')